# Pulling the data

### 1. Pick 6–8 signals max

### 2. Make each one represent a different force:
- Growth
- Inflation
- Fear
- Energy
- Liquidity
- Future demand

### 3. Force each variable to justify its existence:
- If two say the same thing → kill one


### 4. Then build a composite index
- Z-score standardisation
- Weighted (not equal… think carefully)

#### Instead of tracking the dollar directly, track its impact:

Copper falling + dollar rising → global slowdown signal
Gold rising + dollar rising → fear spike (unusual, powerful)
Oil rising + dollar rising → inflation pressure intensifying

Now your system isn’t just descriptive.

It becomes predictive tension.

In [26]:
# UK Economic Pulse - Building the Core Logic Step by Step

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fredapi import Fred
import yfinance as yf
import openpyxl


# Need pandas to connect to dataframes
# Matplotlib is sufficient for simple visualisations
# Fred will give us the data

import io
import re
import requests
import lxml


In [27]:
# FRED_API_KEY = os.getenv("FRED_API_KEY")  # or paste your key as a string for now
FRED_API_KEY = '0acae73adc29df31d82dce5320829f70'
fred = Fred(api_key=FRED_API_KEY)

# Why is it cleaner to use an environment variable?

In [28]:
# Get GDP, House Prices and Asset Prices from FRED

gdp = fred.get_series("NGDPRSAXDCGBQ")
gdp_us = fred.get_series("GDPC1")
house_prices = fred.get_series("QGBR628BIS")

# Commodity Prices — Gold, Silver, Copper, Natural Gas
# All sourced from COMEX/NYMEX futures via yfinance, daily

gold    = yf.download('GC=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/troy oz
silver  = yf.download('SI=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/troy oz
copper  = yf.download('HG=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/lb
nat_gas = yf.download('NG=F', start='2000-01-01', progress=False)['Close'].squeeze()  # USD/MMBtu

# In many regions, Gas sets the marginal price of electricity.

# Market Indicators — Semiconductors, Uranium, US Dollar
# SOX & DXY: yfinance (daily)
# Trade-Weighted USD: FRED DTWEXBGS (daily, Federal Reserve)

sox     = yf.download('^SOX',     start='2000-01-01', progress=False)['Close'].squeeze()  # PHLX Semiconductor Index
uranium = yf.download('URA',      start='2009-01-01', progress=False)['Close'].squeeze()  # Global X Uranium ETF
dxy     = yf.download('DX-Y.NYB', start='2000-01-01', progress=False)['Close'].squeeze()  # ICE US Dollar Index
twusd   = fred.get_series('DTWEXBGS')                                                      # Nominal Broad Trade-Weighted USD

for s in [gold, silver, copper, nat_gas, sox, uranium, dxy]:
    s.index = pd.to_datetime(s.index)

In [29]:
# Get US Credit Spreads from FRED

series = {
    "us_hy_spread": "BAMLH0A0HYM2",
    "us_ig_spread": "BAMLC0A0CM",
    "us_bbb_spread": "BAMLC0A4CBBB"
}

us_credit_spreads = pd.DataFrame({name: fred.get_series(code) for name, code in series.items()})

In [30]:
# Get Bond Yields from FRED

def get_bond_yields() -> pd.DataFrame:
    series_map = {
        # US Treasuries (daily)
        "US_2Y": "DGS2",
        "US_10Y": "DGS10",
        "US_30Y": "DGS30",
        
        # UK Gilts (monthly via OECD on FRED)
        "UK_10Y": "IRLTLT01GBM156N",
        "UK_3M": "IR3TIB01GBM156N"
    }

    data = {}

    for name, series_id in series_map.items():
        s = fred.get_series(series_id)
        s.name = name
        data[name] = s

    df = pd.concat(data.values(), axis=1)

    # Sort and clean
    df = df.sort_index()

    # Align frequencies → monthly (critical)
    df = df.resample("ME").mean()

    return df

In [31]:
# -----------------------------
# 4) ONS Retail Sales Volume Index
# Sheet: KPSA (Seasonally Adjusted), headers at row 7
# -----------------------------

def get_uk_retail_sales_ons() -> pd.Series:
    url = "https://www.ons.gov.uk/file?uri=/businessindustryandtrade/retailindustry/datasets/retailsalesindexreferencetables/current/mainreferencetables.xlsx"

    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_excel(io.BytesIO(r.content), sheet_name="KPSA", header=6)

    time_col = "Time Period"
    value_col = "All retailing including automotive fuel [note1]"
    df = df[[time_col, value_col]].copy()

    # Truncate before the 'Revision to index numbers' section
    revision_mask = df[time_col].astype(str).str.contains("Revision", case=False, na=False)
    if revision_mask.any():
        df = df.iloc[:revision_mask.values.argmax()]

    # Drop blank / metadata rows
    df = df[df[time_col].notna()]
    df = df[df[time_col].astype(str).str.strip().ne("")]

    # Parse 'yyyy-mmm' (e.g. '1996 Jan') to datetime
    date_str = df[time_col].astype(str).str.strip().str.replace("-", " ")
    df["date"] = pd.to_datetime(date_str, format="%Y %b", errors="coerce")

    df["value"] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=["date", "value"])

    # Sort chronologically on datetime BEFORE converting to string
    df = df.sort_values("date")
    return df.set_index("date")["value"]

In [32]:
# -----------------------------
# 1) ONS CPI (index level)
# Series: D7BT = CPI INDEX 00: ALL ITEMS 2015=100
# Source dataset: MM23
# -----------------------------

def get_uk_cpi_index_ons() -> pd.Series:
    url = "https://www.ons.gov.uk/generator?format=csv&uri=/economy/inflationandpriceindices/timeseries/d7bt/mm23"
    
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))

    # Rename the two columns into something sane
    df.columns = ["raw_date", "raw_value"]

    # Keep only monthly rows like '1988 JAN'
    monthly_mask = df["raw_date"].astype(str).str.match(r"^\d{4} [A-Z]{3}$")
    df = df.loc[monthly_mask].copy()

    # Rename
    df = df.rename(columns={
        "raw_date": "date",
        "raw_value": "cpi_index"
    })

    # Parse fields
    df["date"] = pd.to_datetime(df["date"], format="%Y %b", errors="coerce")
    df["cpi_index"] = pd.to_numeric(df["cpi_index"], errors="coerce")

    # Clean up
    df = df.dropna(subset=["date", "cpi_index"]).sort_values("date")
    df = df.set_index("date")

    s = df["cpi_index"]
    s.name = "cpi_index"
    return s

In [33]:
# -----------------------------
# 2) ONS Nominal Wages (AWE Total Pay)
# Series: KAB9
# Source dataset: LMS
# -----------------------------

def get_uk_nominal_wages_ons() -> pd.Series:
    url = "https://www.ons.gov.uk/generator?format=csv&uri=/employmentandlabourmarket/peopleinwork/earningsandworkinghours/timeseries/kab9/lms"
    
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(io.StringIO(r.text))

    df.columns = ["raw_date", "raw_value"]

    # Monthly rows like '2000 JAN'
    monthly_mask = df["raw_date"].astype(str).str.match(r"^\d{4} [A-Z]{3}$")
    df = df.loc[monthly_mask].copy()

    df = df.rename(columns={
        "raw_date": "date",
        "raw_value": "nominal_wage"
    })

    df["date"] = pd.to_datetime(df["date"], format="%Y %b", errors="coerce")
    df["nominal_wage"] = pd.to_numeric(df["nominal_wage"], errors="coerce")

    df = df.dropna(subset=["date", "nominal_wage"]).sort_values("date")
    df = df.set_index("date")

    s = df["nominal_wage"]
    s.name = "nominal_wage"
    return s

def get_uk_real_wages(cpi_index: pd.Series, wages: pd.Series) -> pd.Series:
    # Align dates
    df = pd.concat([cpi_index, wages], axis=1).dropna()

    # Real wage index (base-adjusted)
    df["real_wage"] = df["nominal_wage"] / df["cpi_index"] * 100

    s = df["real_wage"]
    s.name = "real_wage"
    return s

In [36]:
# -----------------------------
# 3) Bank of England policy rate
# Pull from the official Bank Rate history page
# -----------------------------

def get_boe_policy_rate(local_fallback_path=None) -> pd.Series:
    url = "https://www.bankofengland.co.uk/boeapps/database/Bank-Rate.asp"

    session = requests.Session()
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/123.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-GB,en;q=0.9",
        "Referer": "https://www.bankofengland.co.uk/boeapps/database/",
        "Connection": "keep-alive",
    })

    # First hit the database home page to look less like a bot
    try:
        warmup = session.get(
            "https://www.bankofengland.co.uk/boeapps/database/",
            timeout=30
        )
        warmup.raise_for_status()
        time.sleep(1.0)
    except Exception:
        pass

    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()

        tables = pd.read_html(io.StringIO(r.text))
        if not tables:
            raise ValueError("No tables found on Bank Rate page.")

        df = tables[0].copy()
        df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]

        if "date_changed" not in df.columns or "rate" not in df.columns:
            raise ValueError(f"Unexpected columns on Bank Rate page: {list(df.columns)}")

        df["date_changed"] = pd.to_datetime(df["date_changed"], format="%d %b %y", errors="coerce")
        df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

        df = df.dropna(subset=["date_changed", "rate"]).sort_values("date_changed")
        df = df.set_index("date_changed")

        s = df["rate"]
        s.name = "policy_rate"
        return s

    except requests.HTTPError as e:
        if local_fallback_path is None:
            raise RuntimeError(
                f"BoE request blocked with HTTP {e.response.status_code}. "
                "Download the Bank Rate history CSV manually from the BoE page and pass its path "
                "as local_fallback_path."
            ) from e

        # Fallback: read a manually downloaded BoE CSV
        raw = pd.read_csv(local_fallback_path)

        # Adjust if your CSV comes with slightly different column names
        raw.columns = [str(c).strip().lower().replace(" ", "_") for c in raw.columns]

        # Try common variants
        date_col = next((c for c in raw.columns if c in ["date_changed", "date"]), None)
        rate_col = next((c for c in raw.columns if c in ["rate", "value"]), None)

        if date_col is None or rate_col is None:
            raise ValueError(
                f"Could not identify date/rate columns in fallback CSV: {list(raw.columns)}"
            )

        df = raw[[date_col, rate_col]].copy()
        df.columns = ["date_changed", "rate"]

        df["date_changed"] = pd.to_datetime(df["date_changed"], errors="coerce", dayfirst=True)
        df["rate"] = pd.to_numeric(df["rate"], errors="coerce")

        df = df.dropna(subset=["date_changed", "rate"]).sort_values("date_changed")
        df = df.set_index("date_changed")

        s = df["rate"]
        s.name = "policy_rate"
        return s

# ----------------------------- 
# 4) Convert policy rate changes to month-end series 
# -----------------------------

def make_policy_rate_monthly(policy_rate_changes: pd.Series, end_date=None) -> pd.Series:
    monthly = policy_rate_changes.resample("ME").last()

    if end_date is None:
        end_date = pd.Timestamp.today().to_period("M").to_timestamp("M")

    full_index = pd.date_range(start=monthly.index.min(), end=end_date, freq="ME")

    monthly = monthly.reindex(full_index).ffill()
    monthly.name = "policy_rate"
    return monthly

In [37]:
# -----------------------------
# 5) Pull everything together
# -----------------------------

# Economic Activity

gdp = fred.get_series("NGDPRSAXDCGBQ")
gdp_us = fred.get_series("GDPC1")
house_prices = fred.get_series("QGBR628BIS")
retail_sales = get_uk_retail_sales_ons()

# Inflation Pressure

cpi_index = get_uk_cpi_index_ons()
wages = get_uk_nominal_wages_ons()
real_wages = get_uk_real_wages(cpi_index, wages)

# Financial Conditions

policy_rate_changes = get_boe_policy_rate()
policy_rate_monthly = make_policy_rate_monthly(policy_rate_changes)
bond_yields = get_bond_yields() # df
us_credit_spreads = pd.DataFrame({name: fred.get_series(code) for name, code in series.items()}) # df

# Asset prices

oil = fred.get_series("DCOILBRENTEU")

# Missing variables



In [38]:
# Inspect the frequency

for name, s in {

    # Economic Activity
    "GDP (UK)":           gdp,
    "GDP (US)":           gdp_us,
    "House Prices (UK)":  house_prices,
    "Retail Sales (UK)":  retail_sales,

    # Inflation Pressure
    "CPI (UK)":           cpi_index,
    "Real Wages (UK)":    real_wages,

    # Financial Conditions
    "Base Rate (UK)":             policy_rate_monthly,
    "Bond Yields (US 2Y)":        bond_yields["US_2Y"],
    "Bond Yields (US 10Y)":       bond_yields["US_10Y"],
    "Bond Yields (US 30Y)":       bond_yields["US_30Y"],
    "Bond Yields (UK 10Y)":       bond_yields["UK_10Y"],
    "Bond Yields (UK 3M)":        bond_yields["UK_3M"],
    "US High Yield Spread":       us_credit_spreads["us_hy_spread"],
    "US IG Spread":               us_credit_spreads["us_ig_spread"],
    "US BBB Spread":              us_credit_spreads["us_bbb_spread"],

    # Commodity Prices
    "Oil":                oil,
    "Gold":               gold,
    "Silver":             silver,
    "Copper":             copper,
    "Natural Gas":        nat_gas,

    # Market Indicators
    "SOX (Semiconductors)":       sox,
    "Uranium (URA ETF)":          uranium,
    "DXY (US Dollar Index)":      dxy,
    "Trade-Weighted USD":         twusd,

}.items():
    print(name)
    print("Start:", s.index.min())
    print("End:  ", s.index.max())
    print("Missing:", s.isna().sum())
    print("Inferred freq:", pd.infer_freq(s.index[:20]) if len(s) > 20 else pd.infer_freq(s.index))
    print("-" * 40)

GDP (UK)
Start: 1955-01-01 00:00:00
End:   2025-10-01 00:00:00
Missing: 0
Inferred freq: QS-OCT
----------------------------------------
GDP (US)
Start: 1947-01-01 00:00:00
End:   2025-10-01 00:00:00
Missing: 0
Inferred freq: QS-OCT
----------------------------------------
House Prices (UK)
Start: 1968-04-01 00:00:00
End:   2025-10-01 00:00:00
Missing: 0
Inferred freq: QS-OCT
----------------------------------------
Retail Sales (UK)
Start: 1996-01-01 00:00:00
End:   2026-02-01 00:00:00
Missing: 0
Inferred freq: MS
----------------------------------------
CPI (UK)
Start: 1988-01-01 00:00:00
End:   2026-02-01 00:00:00
Missing: 0
Inferred freq: MS
----------------------------------------
Real Wages (UK)
Start: 2000-01-01 00:00:00
End:   2026-01-01 00:00:00
Missing: 0
Inferred freq: MS
----------------------------------------
Base Rate (UK)
Start: 1975-01-31 00:00:00
End:   2026-04-30 00:00:00
Missing: 0
Inferred freq: ME
----------------------------------------
Bond Yields (US 2Y)
Start:

In [41]:
# Economic Activity
gdp_monthly          = gdp.resample('ME').ffill()
gdp_us_monthly       = gdp_us.resample('ME').ffill()
house_prices_monthly = house_prices.resample('ME').ffill()
retail_sales_monthly = retail_sales.resample('ME').last()

# Inflation Pressure
cpi_monthly          = cpi_index.resample('ME').last()
real_wages_monthly   = real_wages.resample('ME').last()

# Financial Conditions
policy_rate_monthly        = policy_rate_monthly.copy()
bond_yields_monthly        = bond_yields.resample('ME').mean()        # df: US_2Y, US_10Y, US_30Y, UK_10Y, UK_3M
us_credit_spreads_monthly  = us_credit_spreads.resample('ME').mean()  # df: us_hy_spread, us_ig_spread, us_bbb_spread

# Commodity Prices
oil_monthly          = oil.resample('ME').mean()
gold_monthly         = gold.resample('ME').mean()
silver_monthly       = silver.resample('ME').last()
copper_monthly       = copper.resample('ME').last()
nat_gas_monthly      = nat_gas.resample('ME').mean()

# Market Indicators
sox_monthly          = sox.resample('ME').mean()
uranium_monthly      = uranium.resample('ME').mean()
dxy_monthly          = dxy.resample('ME').mean()
twusd_monthly        = twusd.resample('ME').mean()

# ffill()  → GDP is quarterly, carry forward until next reading
# mean()   → daily series: average over the month
# last()   → monthly point-in-time series: take end-of-month value
# copy()   → already month-end, no resampling needed

print(gdp_monthly.tail())

2025-06-30    705117.0
2025-07-31    705681.0
2025-08-31    705681.0
2025-09-30    705681.0
2025-10-31    706067.0
Freq: ME, dtype: float64


In [43]:
absolute_values = pd.DataFrame({

    # Economic Activity
    "gdp":            gdp_monthly,
    "gdp_us":         gdp_us_monthly,
    "house_prices":   house_prices_monthly,
    "retail_sales":   retail_sales_monthly,

    # Inflation Pressure
    "cpi":            cpi_monthly,
    "real_wages":     real_wages_monthly,

    # Financial Conditions
    "policy_rate":    policy_rate_monthly,
    "yield_us_2y":    bond_yields_monthly["US_2Y"],
    "yield_us_10y":   bond_yields_monthly["US_10Y"],
    "yield_us_30y":   bond_yields_monthly["US_30Y"],
    "yield_uk_10y":   bond_yields_monthly["UK_10Y"],
    "yield_uk_3m":    bond_yields_monthly["UK_3M"],
    "spread_us_hy":   us_credit_spreads_monthly["us_hy_spread"],
    "spread_us_ig":   us_credit_spreads_monthly["us_ig_spread"],
    "spread_us_bbb":  us_credit_spreads_monthly["us_bbb_spread"],

    # Commodity Prices
    "oil":            oil_monthly,
    "gold":           gold_monthly,
    "silver":         silver_monthly,
    "copper":         copper_monthly,
    "nat_gas":        nat_gas_monthly,

    # Market Indicators
    "sox":            sox_monthly,
    "uranium":        uranium_monthly,
    "dxy":            dxy_monthly,
    "twusd":          twusd_monthly,

})

absolute_values.tail(20)

,gdp,gdp_us,house_prices,retail_sales,cpi,real_wages,policy_rate,yield_us_2y,yield_us_10y,yield_us_30y,...,spread_us_bbb,oil,gold,silver,copper,nat_gas,sox,uranium,dxy,twusd
2024-09-30,696752.0,23478.570,112.2059,100.7,134.2,524.590164,5.00,3.622500,3.723500,4.044500,...,1.198571,74.016667,2572.869983,31.164000,4.4960,2.409250,4929.773999,24.262762,101.018500,121.884855
2024-10-31,699094.0,23586.542,111.4854,100.1,135.0,522.222222,5.00,3.972273,4.095455,4.375455,...,1.078696,75.632609,2688.904350,32.653000,4.3135,2.576957,5198.669115,28.842910,103.311305,123.595291
2024-11-30,699094.0,23586.542,111.4854,100.2,135.1,523.316062,4.75,4.257368,4.355789,4.536842,...,1.011364,74.345238,2649.815015,30.684999,4.0805,2.982000,5014.445996,29.083347,105.843500,126.311337
2024-12-31,699094.0,23586.542,111.4854,100.1,135.6,525.811209,4.75,4.225714,4.391429,4.580476,...,1.002857,73.859500,2643.580950,28.940001,3.9860,3.406619,5058.389509,27.406016,107.190000,127.575781
2025-01-31,703655.0,23548.210,111.5402,99.2,135.4,526.587888,4.75,4.271905,4.629048,4.853810,...,1.020909,79.270455,2722.947626,32.127998,4.2620,3.721381,5147.944531,27.110816,108.609047,128.836952
2025-02-28,703655.0,23548.210,111.5402,101.0,136.0,527.941176,4.50,4.210000,4.451053,4.680526,...,1.022000,75.438000,2897.115800,31.219000,4.5145,3.740947,5053.005217,26.218416,107.353157,127.875832
2025-03-31,703655.0,23548.210,111.5402,102.1,136.5,531.135531,4.50,3.970000,4.280476,4.598095,...,1.137143,72.732857,2986.409528,34.457001,5.0200,4.137476,4532.955264,23.054681,104.187143,126.243043
2025-04-30,705117.0,23770.976,108.8559,101.7,138.2,520.984081,4.50,3.777619,4.279048,4.710952,...,1.376190,68.134500,3211.152367,32.530998,4.5600,3.425762,3993.904250,21.852089,100.717619,124.140200
2025-05-31,705117.0,23770.976,108.8559,100.4,138.4,522.398844,4.25,3.920000,4.423810,4.904762,...,1.215217,64.453000,3280.738095,32.891998,4.6525,3.460810,4690.768578,27.148766,100.083181,122.342162
2025-06-30,705117.0,23770.976,108.8559,100.9,138.9,521.958243,4.25,3.889000,4.383500,4.891000,...,1.103810,71.444762,3350.939990,35.852001,5.0300,3.643650,5230.926001,34.433743,98.361000,120.593870


In [46]:
# write your code below

absolute_values.to_csv('/Users/ornettematthews/Documents/Data/Economics/absolute_values.csv')


In [47]:
signals = pd.DataFrame(index=absolute_values.index)

# --- Economic Activity ---
# YoY % change captures growth momentum vs. same period last year
signals["gdp_growth_yoy"]    = absolute_values["gdp"].pct_change(12, fill_method=None) * 100
signals["gdp_growth_yoy_us"] = absolute_values["gdp_us"].pct_change(12, fill_method=None) * 100
signals["house_prices_yoy"]  = absolute_values["house_prices"].pct_change(12, fill_method=None) * 100
signals["retail_sales_yoy"]  = absolute_values["retail_sales"].pct_change(12, fill_method=None) * 100

# --- Inflation Pressure ---
signals["cpi_yoy"]         = absolute_values["cpi"].pct_change(12, fill_method=None) * 100
signals["real_wages_yoy"]  = absolute_values["real_wages"].pct_change(12, fill_method=None) * 100

# --- Financial Conditions ---
# Rates and spreads are already in level form — no transformation needed
signals["policy_rate"]     = absolute_values["policy_rate"]
signals["yield_us_2y"]     = absolute_values["yield_us_2y"]
signals["yield_us_10y"]    = absolute_values["yield_us_10y"]
signals["yield_us_30y"]    = absolute_values["yield_us_30y"]
signals["yield_uk_10y"]    = absolute_values["yield_uk_10y"]
signals["yield_uk_3m"]     = absolute_values["yield_uk_3m"]
signals["spread_us_hy"]    = absolute_values["spread_us_hy"]
signals["spread_us_ig"]    = absolute_values["spread_us_ig"]
signals["spread_us_bbb"]   = absolute_values["spread_us_bbb"]

# Derived: yield curve shape and cross-country divergence
# Negative = inverted curve = historically precedes recession
signals["us_yield_curve"]     = absolute_values["yield_us_10y"] - absolute_values["yield_us_2y"]
signals["uk_yield_curve"]     = absolute_values["yield_uk_10y"] - absolute_values["yield_uk_3m"]
signals["uk_us_10y_spread"]   = absolute_values["yield_uk_10y"] - absolute_values["yield_us_10y"]

# --- Commodity Prices ---
# YoY % change captures price pressure and demand signals
signals["oil_yoy"]     = absolute_values["oil"].pct_change(12, fill_method=None) * 100
signals["gold_yoy"]    = absolute_values["gold"].pct_change(12, fill_method=None) * 100
signals["silver_yoy"]  = absolute_values["silver"].pct_change(12, fill_method=None) * 100
signals["copper_yoy"]  = absolute_values["copper"].pct_change(12, fill_method=None) * 100  # "Dr. Copper" — global demand barometer
signals["nat_gas_yoy"] = absolute_values["nat_gas"].pct_change(12, fill_method=None) * 100

# --- Market Indicators ---
# Equity indices and USD: YoY % change captures trend and momentum
signals["sox_yoy"]     = absolute_values["sox"].pct_change(12, fill_method=None) * 100
signals["uranium_yoy"] = absolute_values["uranium"].pct_change(12, fill_method=None) * 100
signals["dxy_yoy"]     = absolute_values["dxy"].pct_change(12, fill_method=None) * 100
signals["twusd_yoy"]   = absolute_values["twusd"].pct_change(12, fill_method=None) * 100

signals.tail(10)

,gdp_growth_yoy,gdp_growth_yoy_us,house_prices_yoy,retail_sales_yoy,cpi_yoy,real_wages_yoy,policy_rate,yield_us_2y,yield_us_10y,yield_us_30y,...,uk_us_10y_spread,oil_yoy,gold_yoy,silver_yoy,copper_yoy,nat_gas_yoy,sox_yoy,uranium_yoy,dxy_yoy,twusd_yoy
2025-07-31,1.281518,2.335168,-1.093615,1.794616,3.886398,1.120558,4.25,3.882273,4.391818,4.921818,...,0.200582,-16.572208,39.629713,27.013685,4.098565,49.687946,3.737611,39.145056,-6.316507,-2.650090
2025-08-31,1.281518,2.335168,-1.093615,0.792864,3.723008,1.390226,4.00,3.703810,4.264762,4.868571,...,0.372138,-15.537553,36.700491,39.918553,9.010853,38.496042,15.375840,59.667102,-3.886725,-1.662047
2025-09-30,1.281518,2.335168,-1.093615,2.085402,3.800298,0.444185,4.00,3.568571,4.120476,4.742857,...,0.568024,-8.148451,43.120329,48.418043,6.872776,24.925262,22.802751,77.367490,-3.299183,-1.522579
2025-10-31,0.997434,1.989300,-0.881640,1.898102,3.555556,1.360606,4.00,3.521364,4.061818,4.636818,...,0.510282,-14.661838,50.409735,46.981900,17.433636,30.124852,31.494826,74.335866,-4.400377,-2.292556
2025-11-30,NaN,NaN,NaN,1.497006,3.256847,1.366266,4.00,3.550000,4.093889,4.701667,...,0.404611,-14.188183,54.052858,83.953072,27.080014,49.807615,36.744466,51.122132,-5.759570,-3.874253
2025-12-31,NaN,NaN,NaN,1.698302,3.318584,-0.225545,3.75,3.500909,4.143182,4.800909,...,0.339418,-15.319917,63.076493,142.342783,41.244356,28.990862,41.412121,60.497111,-8.067238,-5.790643
2026-01-31,NaN,NaN,NaN,4.737903,3.028065,1.008712,3.75,3.537000,4.213500,4.838500,...,0.237500,-15.980826,73.740395,143.681539,38.362265,14.687264,52.024551,97.107680,-9.575674,-7.456822
2026-02-28,NaN,NaN,NaN,2.376238,3.014706,NaN,3.75,3.471579,4.125789,4.762632,...,0.306611,-6.032769,72.966397,196.876901,33.004756,-16.351052,60.675982,103.625453,-9.219938,-7.796478
2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,3.75,3.714545,4.245909,4.845000,...,0.454791,41.799112,62.616374,116.762924,11.304783,-26.540638,70.852708,115.189985,-4.515788,-5.008742
2026-04-30,NaN,NaN,NaN,NaN,NaN,NaN,3.75,3.795833,4.310000,4.898333,...,NaN,82.175382,48.141523,151.581591,34.089915,-20.944593,117.068548,136.056764,-1.807150,-3.451491
